In [1]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnRewardThreshold
import os

# 1. 环境配置
env_id = "Reacher-v4"
n_envs = 16  # 并行环境数，充分利用多核 CPU
log_dir = "./logs/reacher_ppo/"
os.makedirs(log_dir, exist_ok=True)

# 创建并行环境
envs = make_vec_env(env_id, n_envs=n_envs)

# 2. 定义模型
# 针对 5060 显卡，我们显式指定使用 cuda
model = PPO(
    policy="MlpPolicy",
    env=envs,
    learning_rate=3e-4,
    n_steps=2048,     # 每次更新前的步数
    batch_size=512,    # 显存充足，可以适当调大
    n_epochs=10,      # 每次更新时优化器运行的次数
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    verbose=1,
    tensorboard_log=log_dir,
    device="cuda"     # 强制使用 NVIDIA 显卡
)

# 3. 设置评估指标与回调
# 每 10000 步评估一次，如果平均分达到 -3.5 (MuJoCo Reacher 很难达到正分，通常-3.5就很完美了)，则停止
eval_env = gym.make(env_id)
stop_callback = StopTrainingOnRewardThreshold(reward_threshold=-3.5, verbose=1)
eval_callback = EvalCallback(
    eval_env, 
    callback_on_new_best=stop_callback,
    eval_freq=10000, 
    log_path=log_dir, 
    best_model_save_path=log_dir,
    deterministic=True
)

# 4. 开始训练
print("开始训练...")
model.learn(total_timesteps=1_000_000, callback=eval_callback)

# 5. 保存最终模型
model.save("ppo_reacher_final")

# 6. 最终评估
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10)
print(f"训练完成！平均奖励: {mean_reward:.2f} +/- {std_reward:.2f}")

c:\Users\QunZ\miniconda3\envs\mujoco\lib\site-packages\gymnasium\envs\registration.py:512: DeprecationWarning: WARN: The environment Reacher-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


Using cuda device
开始训练...
Logging to ./logs/reacher_ppo/PPO_1


c:\Users\QunZ\miniconda3\envs\mujoco\lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
c:\Users\QunZ\miniconda3\envs\mujoco\lib\site-packages\gymnasium\envs\registration.py:512: DeprecationWarning: WARN: The environment Reacher-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -61.2    |
| time/              |          |
|    fps             | 8660     |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 32768    |
---------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 50         |
|    ep_rew_mean          | -58.5      |
| time/                   |            |
|    fps                  | 6492       |
|    iterations           | 2          |
|    time_elapsed         | 10         |
|    total_timesteps      | 65536      |
| train/                  |            |
|    approx_kl            | 0.00960663 |
|    clip_fraction        | 0.0432     |
|    clip_range           | 0.2        |
|    entropy_loss         | -2.76      |
|    explained_variance   | -0.00261   |
|    learning_rate        | 0.0003     |
|   

c:\Users\QunZ\miniconda3\envs\mujoco\lib\site-packages\stable_baselines3\common\evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


New best mean reward!
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -50.3    |
| time/              |          |
|    fps             | 5712     |
|    iterations      | 5        |
|    time_elapsed    | 28       |
|    total_timesteps | 163840   |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 50          |
|    ep_rew_mean          | -47.6       |
| time/                   |             |
|    fps                  | 5664        |
|    iterations           | 6           |
|    time_elapsed         | 34          |
|    total_timesteps      | 196608      |
| train/                  |             |
|    approx_kl            | 0.020919444 |
|    clip_fraction        | 0.163       |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.14       |
|    explained_variance   | 0.764       |
|    lea

In [ ]:
model = PPO.load("ppo_reacher_final")
env = gym.make("Reacher-v4", render_mode="human")
obs, _ = env.reset()
for _ in range(1000):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    env.render()
    if terminated or truncated:
        obs, _ = env.reset()

: 